In [ ]:
import hvplot.pandas  # noqa
import pandas as pd
import numpy as np, os, time
import tensorflow as tf
from tensorflow import keras
from argparse import Namespace
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import tmodel
import hvplot as hv
tf.config.set_visible_devices([], 'GPU')             # disable GPU for this notebook
hv.extension('bokeh')
use_current = False

isig = 0
feature_type=0
signal_indices =  [2,20,24,35,42,47,52,56,64,69,79,84,99]
signal_index = signal_indices[isig]
args: Namespace = tmodel.load_args(signal_index,feature_type)

In [ ]:
data=tmodel.get_demo_data()
signals = data['signals']
T: np.ndarray = data['times'][args.signal]
X: np.ndarray = tmodel.get_features( T, args )
Y: np.ndarray = signals[args.signal]
validation_split = int(0.8*X.shape[0])

strategy = tf.distribute.MirroredStrategy()
print(f"Number of devices: {strategy.num_replicas_in_sync}")
model = tmodel.create_streams_model( X.shape[1], dropout_frac=args.dropout_frac, n_streams=args.nstreams )
model.compile( optimizer=tf.keras.optimizers.Adam( learning_rate=args.learning_rate ), loss=args.loss )

In [ ]:
ckp_file = tmodel.get_ckp_file( args, "latest" )
model.load_weights(ckp_file)
P=model.predict( X )
Ttrain=T[:validation_split]
Ptrain=P[:validation_split,0]
Tval=T[validation_split:]
Pval=P[validation_split:,0]

In [ ]:
title=f'Signal {args.signal} (ftype={args.feature_type}): nfeatures={args.nfeatures}'

target       = pd.DataFrame({ 't': T,      's': Y })
train_result = pd.DataFrame({ 't': Ttrain, 's': Ptrain })
val_result   = pd.DataFrame({ 't': Tval,   's': Pval })

pargs = dict( x='t', y='s', ylim=(Y.min()*.98,Y.max()*1.02) )
fargs = dict( legend_position='right', show_legend=True, title=title, xlabel='Time', height=500, width=1500 )
plot1 =       target.hvplot.line( **pargs, label='Target',     color='red'   )
plot2 = train_result.hvplot.line( **pargs, label='Train',      color='blue'  )
plot3 =   val_result.hvplot.line( **pargs, label='Validation', color='green' )

overlay_plot = ( plot1 * plot2 * plot3 ).opts( **fargs )
overlay_plot